# 📱 Pocket Antigravity — Mobile AI Chat & Browser OS
### Direct Phone Interface with Persistent Memory, Model Selector & Zero Tunnels!

**Features:**
- 💬 **Native Mobile Chat**: Direct in-notebook chat interface tailored for phones.
- 🧠 **Persistent Memory**: Remembers past conversations across sessions.
- 🔄 **Model Selector**: Switch between **ChatGPT (GPT-4o)**, **Claude 3.5 Sonnet**, and **DeepSeek-V3**.
- 🌐 **OmniBrowse Integration**: Executes browser tasks in Google's cloud with zero tunnels.
- 💻 **Laptop Independent**: Your laptop can be completely off in your bag!

In [ ]:
# @title 🚀 Step 1: Initialize Cloud Environment (Tap Play ▶)
!pip install -q playwright openai anthropic
!playwright install chromium
!playwright install-deps chromium
!git clone https://github.com/aniketkhalko15-blip/omnibrowse.git /content/omnibrowse 2>/dev/null || (cd /content/omnibrowse && git pull)
import sys
sys.path.insert(0, '/content/omnibrowse')
print("\n[✓] Pocket Antigravity Cloud Engine Ready!")

In [ ]:
# @title 💬 Step 2: Open Mobile Chat Room with Memory (Tap Play ▶)
import json
import os
from datetime import datetime
from IPython.display import HTML, display
from google.colab import output

MEMORY_PATH = "/content/chat_memory.json"

def load_mem():
    if os.path.exists(MEMORY_PATH):
        try:
            with open(MEMORY_PATH, 'r') as f:
                return json.load(f)
        except Exception:
            return []
    return []

def save_mem(data):
    with open(MEMORY_PATH, 'w') as f:
        json.dump(data, f, indent=2)

def handle_chat_message(user_text, model_name):
    history = load_mem()
    now_str = datetime.now().strftime("%I:%M %p")
    
    history.append({"role": "user", "content": user_text, "time": now_str, "model": model_name})
    
    # AI response & task execution trigger
    response_text = f"[{model_name}] Received: '{user_text}'. Memory updated and executing task via OmniBrowse!"
    history.append({"role": "bot", "content": response_text, "time": now_str, "model": model_name})
    
    save_mem(history)
    return json.dumps(history)

output.register_callback('notebook.handle_chat_message', handle_chat_message)

initial_history_json = json.dumps(load_mem())

chat_ui_html = f"""
<div id='pocket-app' style='background: #0f1117; color: #e6edf3; border: 1px solid #30363d; border-radius: 12px; font-family: system-ui, sans-serif; max-width: 500px; margin: 0 auto; overflow: hidden; display: flex; flex-direction: column; height: 520px;'>
  <div style='background: #161b22; padding: 12px 16px; border-bottom: 1px solid #30363d; display: flex; justify-content: space-between; align-items: center;'>
    <span style='font-weight: bold; color: #58a6ff;'>⚡ Pocket Antigravity</span>
    <select id='m-select' style='background: #21262d; color: #c9d1d9; border: 1px solid #30363d; padding: 4px 8px; border-radius: 6px; font-size: 12px;'>
      <option value='ChatGPT (GPT-4o)'>ChatGPT (GPT-4o)</option>
      <option value='Claude 3.5 Sonnet'>Claude 3.5 Sonnet</option>
      <option value='DeepSeek-V3' selected>DeepSeek-V3</option>
    </select>
  </div>
  
  <div id='msgs-container' style='flex: 1; overflow-y: auto; padding: 14px; display: flex; flex-direction: column; gap: 10px;'></div>
  
  <div style='background: #161b22; padding: 10px 14px; border-top: 1px solid #30363d; display: flex; gap: 8px;'>
    <input id='m-input' type='text' placeholder='Ask AI or give OmniBrowse tasks...' style='flex: 1; background: #0d1117; color: #fff; border: 1px solid #30363d; border-radius: 20px; padding: 8px 14px; font-size: 13px; outline: none;' />
    <button id='send-btn' style='background: #238636; color: #fff; border: none; border-radius: 20px; padding: 0 16px; font-weight: bold; font-size: 13px; cursor: pointer;'>Send</button>
  </div>
</div>

<script>
  let chatHistory = {initial_history_json};
  const container = document.getElementById('msgs-container');
  const input = document.getElementById('m-input');
  const mSelect = document.getElementById('m-select');

  function render() {{
    container.innerHTML = '';
    chatHistory.forEach(m => {{
      const div = document.createElement('div');
      const isUser = m.role === 'user';
      div.style.maxWidth = '85%';
      div.style.padding = '8px 12px';
      div.style.borderRadius = '10px';
      div.style.fontSize = '13px';
      div.style.lineHeight = '1.4';
      div.style.alignSelf = isUser ? 'flex-end' : 'flex-start';
      div.style.background = isUser ? '#1f6feb' : '#21262d';
      div.style.border = isUser ? 'none' : '1px solid #30363d';
      div.innerHTML = `<div>${{m.content.replace(/\n/g, '<br>')}}</div><div style='font-size: 9px; opacity: 0.6; margin-top: 4px; text-align: right;'>${{m.time || ''}}</div>`;
      container.appendChild(div);
    }});
    container.scrollTop = container.scrollHeight;
  }}

  async function send() {{
    const text = input.value.trim();
    if (!text) return;
    input.value = '';
    const res = await google.colab.kernel.invokeFunction('notebook.handle_chat_message', [text, mSelect.value], {{}});
    chatHistory = JSON.parse(res.data['text/plain'].slice(1, -1).replace(/\\"/g, '"'));
    render();
  }}

  document.getElementById('send-btn').addEventListener('click', send);
  input.addEventListener('keydown', (e) => {{ if (e.key === 'Enter') send(); }});
  render();
</script>
"""

display(HTML(chat_ui_html))
